In [2]:
import requests
import pandas as pd

# -----------------------
# Credentials
# -----------------------
token_url = "https://app1pub.smappee.net/dev/v3/oauth2/token"

client_id = "165804"
client_secret = "ty2HNwm2AF"
username = "Ilias.EsSahili@oracdecor.be"
password = "Traverse4-Yonder-Skylight-Unsubtle-Epidermal!"

# -----------------------
# Get Access Token
# -----------------------
payload = {
    "grant_type": "password",
    "client_id": client_id,
    "client_secret": client_secret,
    "username": username,
    "password": password
}

headers = {
    "Content-Type": "application/x-www-form-urlencoded;charset=UTF-8"
}

token_response = requests.post(token_url, data=payload, headers=headers)
token_response.raise_for_status()

access_token = token_response.json()["access_token"]

In [3]:
print("Access Token:", access_token)

Access Token: e7767cc5-762f-3f08-8e4d-bc0ccc0245ff


In [4]:
import json

url = "https://app1pub.smappee.net/dev/v3/servicelocation"

headers = {
    "Authorization": f"Bearer {access_token}",
    "Accept": "application/json"
}

response = requests.get(url, headers=headers)
response.raise_for_status()

data = response.json()

servicelocations = data["serviceLocations"]

print(json.dumps(servicelocations, indent=4))



[
    {
        "serviceLocationId": 75452,
        "serviceLocationUuid": "01d1d4a4-32b0-4be4-90ee-a0a78920f542",
        "name": "Orac",
        "deviceSerialNumber": "5010005672"
    },
    {
        "serviceLocationId": 80060,
        "serviceLocationUuid": "34666732-62c6-465d-ada6-27fdd30c5ab4",
        "name": "Orac Oostende laadplein",
        "deviceSerialNumber": "5130099474"
    },
    {
        "serviceLocationId": 80071,
        "serviceLocationUuid": "903908ae-172d-493c-956a-5f3e58fbdbb3",
        "name": "Ev Base Orac 1",
        "deviceSerialNumber": "5130005802"
    },
    {
        "serviceLocationId": 80072,
        "serviceLocationUuid": "9200f38f-4f99-4db2-a074-acbd2cd7ca44",
        "name": "Ev Base Orac 2",
        "deviceSerialNumber": "5130008024"
    },
    {
        "serviceLocationId": 80074,
        "serviceLocationUuid": "38b908ae-699f-49fa-bea8-9df869d65a80",
        "name": "Ev Base Orac 3",
        "deviceSerialNumber": "5130008032"
    },
    {
        

In [5]:
import requests

def get_consumption(service_location_id, access_token, aggregation, from_ts, to_ts):

    url = f"https://app1pub.smappee.net/dev/v3/servicelocation/{service_location_id}/consumption"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Accept": "application/json"
    }

    params = {
        "aggregation": aggregation,
        "from": int(from_ts),
        "to": int(to_ts)
    }

    response = requests.get(url, headers=headers, params=params)

    print("REQUEST URL:", response.url)  # 🔥 DEBUG IMPORTANT
    print("STATUS:", response.status_code)
    print("RAW RESPONSE:", response.text[:500])

    response.raise_for_status()

    return response.json()

In [6]:
import requests

def get_servicelocation_info(service_location_id, access_token):

    url = f"https://app1pub.smappee.net/dev/v3/servicelocation/{service_location_id}/info"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Accept": "application/json"
    }

    # Geen params nodig (zoals from_ts of to_ts) voor de info endpoint
    response = requests.get(url, headers=headers)

    print("REQUEST URL:", response.url) 
    print("STATUS:", response.status_code)
    # We printen hier iets meer tekens (1000) omdat de info-JSON vaak best groot is
    print("RAW RESPONSE:", response.text[:1000]) 

    response.raise_for_status()

    return response.json()

In [26]:
import time
import pandas as pd
import matplotlib.pyplot as plt


def get_time_range(period: str):
    """
    period: 'day', 'week', 'month', 'year'
    returns from_ts, to_ts in milliseconds
    """
    to_ts = int(time.time() * 1000)

    if period == "day":
        from_ts = to_ts - (1 * 24 * 60 * 60 * 1000)
    elif period == "week":
        from_ts = to_ts - (7 * 24 * 60 * 60 * 1000)
    elif period == "month":
        from_ts = to_ts - (30 * 24 * 60 * 60 * 1000)
    elif period == "year":
        from_ts = to_ts - (365 * 24 * 60 * 60 * 1000)
    else:
        raise ValueError("Invalid period. Use: day, week, month, year")

    return from_ts, to_ts


def plot_energy(data, period="week"):
    # timestamps bepalen (optioneel, mag weg als je het extern doet)
    from_ts, to_ts = get_time_range(period)

    # data ophalen
    rows = data["consumptions"]

    df = pd.DataFrame(rows)

    # timestamp → datetime
    df["date"] = pd.to_datetime(df["timestamp"], unit="ms")
    df = df.sort_values("date")

    # kWh conversie
    df["consumption_kwh"] = df["consumption"] / 1000
    df["solar_kwh"] = df["solar"] / 1000
    df["gridImport_kwh"] = df["gridImport"] / 1000
    df["gridExport_kwh"] = df["gridExport"] / 1000

    # --- NIEUW: kWh naar kW conversie ---
    # 1. Bereken het tijdsverschil in uren tussen elke rij
    df["duration_hours"] = df["date"].diff().dt.total_seconds() / 3600
    
    # De allereerste rij heeft geen vorige rij om mee te vergelijken (NaN), 
    # dus we kopiëren de tijdsduur van de tweede rij naar de eerste.
    df["duration_hours"] = df["duration_hours"].bfill()

    # 2. Deel de kWh door de uren om het vermogen (kW) te krijgen
    df["consumption_kw"] = df["consumption_kwh"] / df["duration_hours"]
    df["solar_kw"] = df["solar_kwh"] / df["duration_hours"]
    df["gridImport_kw"] = df["gridImport_kwh"] / df["duration_hours"]
    df["gridExport_kw"] = df["gridExport_kwh"] / df["duration_hours"]

    # plot
    plt.figure(figsize=(14, 6))

    # Gebruik hier nu de _kw kolommen!
    plt.plot(df["date"], df["consumption_kw"], label="Consumption (kW)")
    plt.plot(df["date"], df["solar_kw"], label="Solar (kW)")
    plt.plot(df["date"], df["gridImport_kw"], linestyle="dotted", label="Grid Import (kW)")
    plt.plot(df["date"], df["gridExport_kw"], linestyle="dotted", label="Grid Export (kW)")

    plt.title(f"Power overview ({period})")
    plt.xlabel("Date")
    plt.ylabel("kW")  # Aangepast naar kW

    plt.legend()


    # 🔥 clean date formatting (fix van jouw issue)
    plt.gca().xaxis.set_major_locator(mdates.AutoDateLocator())
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%d-%m %H:%M'))
    plt.gcf().autofmt_xdate()

    plt.tight_layout()
    plt.show()

In [8]:
import requests

def get_metering_configuration(service_location_id, access_token):
    # De specifieke URL voor metering configuration
    url = f"https://app1pub.smappee.net/dev/v3/servicelocation/{service_location_id}/meteringconfiguration"

    # Headers met de vereiste authenticatie
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Accept": "application/json"
    }

    # Uitvoeren van de GET request (geen extra params nodig voor deze specifieke call)
    response = requests.get(url, headers=headers)

    # 🔥 DEBUG IMPORTANT (overgenomen uit jouw voorbeeld)
    print("REQUEST URL:", response.url)  
    print("STATUS:", response.status_code)
    print("RAW RESPONSE:", response.text[:500])

    # Foutmelding gooien als de status code geen 200 OK is
    response.raise_for_status()

    # Parsen en teruggeven van de JSON
    return response.json()

# --- Voorbeeld van hoe je deze functie aanroept ---
# service_loc_id = "123"
# token = "JOUW_ACCESS_TOKEN"
# config_data = get_metering_configuration(service_loc_id, token)
# print(config_data)

In [33]:
config_data = get_metering_configuration(service_location_id=87764, access_token=access_token)
print(config_data)

REQUEST URL: https://app1pub.smappee.net/dev/v3/servicelocation/87764/meteringconfiguration
STATUS: 200
RAW RESPONSE: {
  "lon": 2.961313699429526,
  "lat": 51.21851211997366,
  "electricityCost": 0.0,
  "electricityCurrency": "EUR",
  "timezone": "Europe/Brussels",
  "appliances": [
    {
      "id": 2,
      "name": "A.1 (hoofdverdeelbord)",
      "type": "Other",
      "sourceType": "CT"
    },
    {
      "id": 6,
      "name": "A.6 (pompen koelwater/ stofafzuiging)",
      "type": "Other",
      "sourceType": "CT"
    },
    {
      "id": 5,
      "name": "A.5 (extrusie lijn 1)",
      "type": "Other",
   
{'lon': 2.961313699429526, 'lat': 51.21851211997366, 'electricityCost': 0.0, 'electricityCurrency': 'EUR', 'timezone': 'Europe/Brussels', 'appliances': [{'id': 2, 'name': 'A.1 (hoofdverdeelbord)', 'type': 'Other', 'sourceType': 'CT'}, {'id': 6, 'name': 'A.6 (pompen koelwater/ stofafzuiging)', 'type': 'Other', 'sourceType': 'CT'}, {'id': 5, 'name': 'A.5 (extrusie lijn 1)', 'type'

In [12]:
data = get_servicelocation_info(service_location_id=75452, access_token=access_token)

print(data)

REQUEST URL: https://app1pub.smappee.net/dev/v3/servicelocation/75452/info
STATUS: 200
RAW RESPONSE: {
  "lon": 2.9612479567454173,
  "lat": 51.218414306640625,
  "electricityCost": 0.0,
  "electricityCurrency": "EUR",
  "timezone": "Europe/Brussels",
  "appliances": [],
  "actuators": [],
  "sensors": [],
  "monitors": [],
  "channelsConfiguration": {
    "inputChannels": [
      {
        "ctInput": 0,
        "name": "Grid",
        "phase": 0,
        "reversed": false,
        "nilm": false,
        "balanced": false,
        "inputChannelCTType": "CT50_100_200"
      },
      {
        "ctInput": 0,
        "name": "Grid",
        "phase": 1,
        "reversed": false,
        "nilm": false,
        "balanced": false,
        "inputChannelCTType": "CT50_100_200"
      },
      {
        "ctInput": 0,
        "name": "Grid",
        "phase": 2,
        "reversed": false,
        "nilm": false,
        "balanced": false,
        "inputChannelCTType": "CT50_100_200"
      },
      {

In [24]:
def get_solar_production(service_location_id, access_token, from_ts, to_ts):
    # Gebruik de algemene consumption endpoint
    url = f"https://app1pub.smappee.net/dev/v3/servicelocation/{service_location_id}/consumption"
    
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Accept": "application/json"
    }
    
    params = {
        "aggregation": 1, # 1 = per dag (bijvoorbeeld), pas aan naar wens (1, 2, 3, 4)
        "from": int(from_ts),
        "to": int(to_ts)
    }
    
    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()
    
    data = response.json()
    
    # De response bevat een lijst met 'consumptions'. 
    # Elke entry heeft velden zoals 'consumption', 'solar', 'gridImport', etc.
    total_solar_wh = 0
    
    if "consumptions" in data:
        for record in data["consumptions"]:
            # 'solar' geeft de zonneproductie in Wattuur (Wh)
            total_solar_wh += record.get("solar", 0)
            
    print(f"Totale zonneproductie in deze periode: {total_solar_wh} Wh")
    print("RAW RESPONSE:", response.text[:500])
    return data

# Voorbeeldgebruik:


In [ ]:
from_ts_1, to_ts_1 = get_time_range("day")


data = get_solar_production(87793 , access_token, from_ts_1, to_ts_1)


plot_energy(data)

Totale zonneproductie in deze periode: 0.0 Wh
RAW RESPONSE: {"serviceLocationId":87793,"consumptions":[{"timestamp":1780490100000,"consumption":0.0,"solar":0.0,"alwaysOn":0.0,"gridImport":0.0,"gridExport":0.0,"selfConsumption":0.0,"selfSufficiency":0.0,"active":[-3.9,15.9,38.4,251.1,217.0,212.5,276.9,158.2,138.4,219.1,426.9,314.0,156.3,157.6,162.8,434.6,346.2,428.9,107.0,91.0,146.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0],"reactive":[32.5,2.9,2.1,43.9,13.9,10.5,66.5,-8.8,8.0,98.4,86.3,153.0,74.3,66.0,74.6,347.2,508.8,498.6,-30.8,-11.6,-15.9,0.0,0.0,0.0,0.0,0.0,0.0,0


AttributeError: 'dict' object has no attribute 'json'

In [40]:
print(data)

{'serviceLocationId': 87793, 'consumptions': [{'timestamp': 1780490100000, 'consumption': 0.0, 'solar': 0.0, 'alwaysOn': 0.0, 'gridImport': 0.0, 'gridExport': 0.0, 'selfConsumption': 0.0, 'selfSufficiency': 0.0, 'active': [-3.9, 15.9, 38.4, 251.1, 217.0, 212.5, 276.9, 158.2, 138.4, 219.1, 426.9, 314.0, 156.3, 157.6, 162.8, 434.6, 346.2, 428.9, 107.0, 91.0, 146.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 'reactive': [32.5, 2.9, 2.1, 43.9, 13.9, 10.5, 66.5, -8.8, 8.0, 98.4, 86.3, 153.0, 74.3, 66.0, 74.6, 347.2, 508.8, 498.6, -30.8, -11.6, -15.9, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 'voltages': [234.1, 234.7, 235.6], 'lineVoltages': [405.7, 407.9, 406.3], 'phaseVoltages': [234.1, 234.7, 235.6], 'current': [1.7, 0.8, 2.0, 14.3, 12.6, 12.3, 15.0, 9.3, 8.3, 12.7, 23.1, 18.4, 9.3, 9.1, 9.5, 28.8, 33.6, 35.9, 5.7, 4.7, 8.7, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 'currentHarmonics': [[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []], 'voltage

In [ ]:
from_ts, to_ts = get_time_range("day")

data = get_consumption(
    service_location_id=75452,
    access_token=access_token,
    aggregation=2,
    from_ts=from_ts_1,
    to_ts=to_ts_1
)
print(data.json())

REQUEST URL: https://app1pub.smappee.net/dev/v3/servicelocation/75452/consumption?aggregation=2&from=1780488728827&to=1780575128827
STATUS: 200
RAW RESPONSE: {"serviceLocationId":75452,"consumptions":[{"timestamp":1780491600000,"consumption":723474.227,"solar":175475.644,"alwaysOn":291351.632,"gridImport":197047.894,"gridExport":0.0,"selfConsumption":100.0,"selfSufficiency":72.76,"active":[null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null],"reactive":[null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,nu


In [18]:
print(data)

{'serviceLocationId': 75452, 'consumptions': [{'timestamp': 1779721200000, 'consumption': 355661.533, 'solar': 122763.68, 'alwaysOn': 31384.757, 'gridImport': 71.7, 'gridExport': 12700.809, 'selfConsumption': 89.65, 'selfSufficiency': 99.98, 'active': [None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None], 'reactive': [None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None], 'voltages': [None, None, None], 'phaseVoltages': [None, None, None], 'currentHarmonics': [[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []], 'voltageHarmonics': [[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []]}, {'timestamp': 1779724800000, 'consumption': 1096898.177, '

In [ ]:
from_ts, to_ts = get_time_range("day")

data2 = get_consumption(
    service_location_id=80060,
    access_token=access_token,
    aggregation=7,
    from_ts=from_ts,
    to_ts=to_ts
)

print(data2)

plot_energy(data2, period="week")

NameError: name 'get_time_range' is not defined

In [ ]:
def transform_to_db_format(data, service_location_id):
    rows = data["consumptions"]

    df = pd.DataFrame(rows)

    # -----------------------
    # timestamp fix
    # -----------------------
    df["date"] = pd.to_datetime(df["timestamp"], unit="ms")

    # -----------------------
    # kWh conversie
    # -----------------------
    df["consumption_kwh"] = df["consumption"] / 1000
    df["solar_kwh"] = df["solar"] / 1000
    df["grid_import_kwh"] = df["gridImport"] / 1000
    df["grid_export_kwh"] = df["gridExport"] / 1000

    # -----------------------
    # extra kolommen voor DB
    # -----------------------
    df["service_location_id"] = service_location_id
    df["timestamp"] = df["timestamp"]  # blijft bigint

    # -----------------------
    # alleen kolommen voor SQL tabel
    # -----------------------
    df = df[[
        "service_location_id",
        "timestamp",
        "date",
        "consumption_kwh",
        "solar_kwh",
        "grid_import_kwh",
        "grid_export_kwh"
    ]]

    return df

In [ ]:
df = pd.DataFrame(data2["consumptions"])

df["date"] = pd.to_datetime(df["timestamp"], unit="ms")

df["consumption_kwh"] = df["consumption"] / 1000
df["solar_kwh"] = df["solar"] / 1000
df["grid_import_kwh"] = df["gridImport"] / 1000
df["grid_export_kwh"] = df["gridExport"] / 1000


In [ ]:
#Zorg dat alle tabellen waar waarden in staan die met een . gedecimalen zijn, naar een , decimaal omgezet worden.
df["consumption_kwh"] = df["consumption_kwh"].apply(lambda x: str(x).replace(".", ","))
df["solar_kwh"] = df["solar_kwh"].apply(lambda x: str(x).replace(".", ","))
df["grid_import_kwh"] = df["grid_import_kwh"].apply(lambda x: str(x).replace(".", ","))
df["grid_export_kwh"] = df["grid_export_kwh"].apply(lambda x: str(x).replace(".", ","))


In [ ]:

df.to_csv("energy_data_ev_hourly.csv", index=False)